# Package

In [3]:
# ============================================================
# 0) Future & Typing (always first)
# ============================================================
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional

# ============================================================
# 1) Standard Library
# ============================================================
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from dateutil.relativedelta import relativedelta

# ============================================================
# 2) Project Path Setup
# ============================================================
# Notebook located in /notebooks → project root = parent
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# 3) Core Scientific Stack
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# 4) Data Loading & Custom Utilities
# ============================================================
from utils import load_wide_from_feast, build_unrate_exog_dataset
from experiment_utils import to_wide_from_oos, make_mae_dm_pivot

# ============================================================
# 5) Forecasting Framework (MLForecast)
# ============================================================
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ============================================================
# 6) Machine Learning Models
# ============================================================
from sklearn.linear_model import LinearRegression, Ridge
from lightgbm import LGBMRegressor

# ============================================================
# 7) Model Selection & Metrics
# ============================================================
from sklearn.model_selection import GridSearchCV, ParameterSampler, ParameterGrid
from sklearn.metrics import mean_absolute_error

# ============================================================
# 8) Statistical Tests
# ============================================================
from scipy.stats import ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

# ============================================================
# 9) Visualization
# ============================================================
from utilsforecast.plotting import plot_series
from IPython.display import IFrame, display

# ============================================================
# 10) Experiment Tracking (MLflow)
# ============================================================
import mlflow
from mlflow.tracking import MlflowClient

# Importation des données

In [5]:
series_ids = [
    "BUSLOANS","CPIAUCSL","DPCERA3M086SBEA","INDPRO",
    "M2SL","OILPRICEX","RPI","SP500","TB3MS","UNRATE","USREC",
]

START = "1960-01-01"
END   = "2025-08-01"

In [6]:
# ============================================================
# 1) WIDE dataset (df) depuis Feast
# ============================================================
import pandas as pd

LAG = 6  # ✅ tout lag = 12 (aucune variable contemporaine)

df = load_wide_from_feast(
    "stationary_value:value",
    series_ids,
    start=START,
    end=END
)

# ============================================================
# 2) Convert WIDE -> LONG (obligatoire pour build_unrate_exog_dataset)
# ============================================================
df_stationary = (
    df.reset_index()
      .melt(id_vars="date", var_name="series_id", value_name="value")
)

# ============================================================
# 3) Build target + exog + MLForecast format
# ============================================================
df_model, ts_lr, exog_cols = build_unrate_exog_dataset(df_stationary)

# ============================================================
# 4) ✅ TOUTES les exog laggées de 12 mois (aucune contemporaine)
#    - on crée BUSLOANS_lag12, CPIAUCSL_lag12, ...
#    - on supprime BUSLOANS, CPIAUCSL, ... (contemporaines)
# ============================================================
ts_lr = ts_lr.sort_values(["unique_id", "ds"]).copy()

exog_cols_lag12 = []
for c in exog_cols:
    new_c = f"{c}_lag{LAG}"
    ts_lr[new_c] = ts_lr.groupby("unique_id")[c].shift(LAG)
    exog_cols_lag12.append(new_c)

# supprimer les exog contemporaines
ts_lr = ts_lr.drop(columns=exog_cols)

# mettre à jour la liste exog utilisée par MLForecast
exog_cols = exog_cols_lag12

# drop lignes où les lag12 n'existent pas (12 premiers mois)
ts_lr = ts_lr.dropna(subset=["y"] + exog_cols).reset_index(drop=True)

# ============================================================
# 5) Prints / checks
# ============================================================
print("df (wide) shape:", df.shape)
print("df_stationary (long) shape:", df_stationary.shape)
print("df_model shape:", df_model.shape)
print("ts_lr shape (after lag12):", ts_lr.shape)
print("Exog cols (lag12):", exog_cols)

print("\nPreview:")
print(ts_lr[["unique_id", "ds", "y"] + exog_cols[:5]].head(15))

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df (wide) shape: (788, 11)
df_stationary (long) shape: (8668, 3)
df_model shape: (788, 12)
ts_lr shape (after lag12): (782, 13)
Exog cols (lag12): ['BUSLOANS_lag6', 'CPIAUCSL_lag6', 'DPCERA3M086SBEA_lag6', 'INDPRO_lag6', 'M2SL_lag6', 'OILPRICEX_lag6', 'RPI_lag6', 'SP500_lag6', 'TB3MS_lag6', 'USREC_lag6']

Preview:
   unique_id         ds    y  BUSLOANS_lag6  CPIAUCSL_lag6  \
0     UNRATE 1960-07-01  0.4       0.011578      -0.006156   
1     UNRATE 1960-08-01  0.4       0.011905      -0.003767   
2     UNRATE 1960-09-01  0.0      -0.008356      -0.005455   
3     UNRATE 1960-10-01  0.4      -0.009098       0.005090   
4     UNRATE 1960-11-01  0.3      -0.000359       0.003383   
5     UNRATE 1960-12-01  1.3       0.014620       0.006777   
6     UNRATE 1961-01-01  1.4      -0.000611      -0.005433   
7     UNRATE 1961-02-01  2.1      -0.016888      -0.004074   
8     UNRATE 1961-03-01  1.

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\ML Experiment\utils.py:171: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


# Model settings

In [7]:
from sklearn.linear_model import LinearRegression, Ridge
from lightgbm import LGBMRegressor
from mlforecast import MLForecast

# -------------------------------------------------
# Registry propre : 1 modèle = 1 MLForecast
# -------------------------------------------------

def make_lr(freq):
    return MLForecast(
        models={"LR": LinearRegression()},
        freq=freq,
        lags=[],
        date_features=[],
    )

def make_ridge(freq, alpha=1.0):
    return MLForecast(
        models={"Ridge": Ridge(alpha=alpha)},
        freq=freq,
        lags=[],
        date_features=[],
    )

def make_lgbm(freq, **params):
    return MLForecast(
        models={"LGBM": LGBMRegressor(**params)},
        freq=freq,
        lags=[],
        date_features=[],
    )

MLF_MODELS = {
    "LR_EXOG_ONLY": make_lr,
    "RIDGE_EXOG_ONLY": make_ridge,
    "LGBM_EXOG_ONLY": make_lgbm,
}

In [8]:
def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    ds_start = _ensure_ms(ds_start)
    ds_end   = _ensure_ms(ds_end)
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

In [9]:
def _prep_monthly_ts(ts, exp_end):
    ts = ts.copy()
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")
    exp_end = _ensure_ms(exp_end)
    return ts[ts["ds"] <= exp_end].copy()

def _slice_cv_block(ts, cutoff_start, n_windows, h):
    cutoff_end = cutoff_start + relativedelta(months=n_windows - 1)
    ds_end = cutoff_end + relativedelta(months=h)
    return ts[ts["ds"] <= ds_end].copy(), cutoff_end, ds_end

# Tuning

In [ ]:
# ============================================================
# Result object (standard)
# ============================================================

@dataclass
class TuneResult:
    best_params: Optional[Dict[str, Any]]
    best_score: float          # score à minimiser (ex: MAE)
    trials: pd.DataFrame       # historique des essais

# ============================================================
# General tuner (agnostic model)
# ============================================================

def tune_model(
    ts_train: pd.DataFrame,
    *,
    objective_fn: Callable[[Dict[str, Any]], float],
    param_space: Dict[str, Iterable[Any]],
    search: str = "random",      # "random" | "grid"
    n_iter: int = 50,            # utilisé si search="random"
    seed: int = 0,
    min_train_n: Optional[int] = None,
    return_trials: bool = True,
) -> TuneResult:
    """
    Tuner générique.
    Minimise objective_fn(params) sur ts_train.
    """

    if min_train_n is not None and len(ts_train) < int(min_train_n):
        return TuneResult(
            best_params=None,
            best_score=float("nan"),
            trials=pd.DataFrame(),
        )

    if search not in ("random", "grid"):
        raise ValueError("search must be 'random' or 'grid'")

    # Générateur d'essais
    if search == "grid":
        sampler = ParameterGrid(param_space)
    else:
        sampler = ParameterSampler(
            param_space,
            n_iter=int(n_iter),
            random_state=int(seed),
        )

    best_params: Optional[Dict[str, Any]] = None
    best_score: float = float("inf")
    rows: List[Dict[str, Any]] = []

    for params in sampler:
        params = dict(params)

        try:
            score = float(objective_fn(params))
        except Exception as e:
            score = float("inf")
            params["_error"] = repr(e)

        rows.append({"score": score, **params})

        if score < best_score:
            best_score = score
            best_params = {
                k: v for k, v in params.items()
                if k != "_error"
            }

    trials = (
        pd.DataFrame(rows).sort_values("score")
        if return_trials else pd.DataFrame()
    )

    return TuneResult(
        best_params=best_params,
        best_score=best_score,
        trials=trials,
    )

In [94]:
# ============================================================
# Nixtla-compatible objective factory
# ============================================================

def make_objective_nixtla_cv(
    ts_train: pd.DataFrame,
    *,
    build_mlf_fn: Callable[[Dict[str, Any]], Any],   # params -> mlf
    run_cv_block_fn: Callable[..., pd.DataFrame],    # ton run_cross_validation_block
    pred_col: str,                                   # ex "LGBM", "Ridge"
    h: int,
    step_size: int,
    tune_cv_windows: int,
    pi_tune=None,                                    # recommandé None pour tuning rapide
    levels_tune=None,                                # ex [95] si pi_tune != None
) -> Callable[[Dict[str, Any]], float]:
    """
    Fabrique une objective_fn compatible tune_model()
    en utilisant ton run_cross_validation_block().
    """

    def objective_fn(params: Dict[str, Any]) -> float:
        mlf = build_mlf_fn(params)

        cv = run_cv_block_fn(
            ts=ts_train,
            mlf=mlf,
            h=int(h),
            step_size=int(step_size),
            partitions=int(tune_cv_windows),
            pi=pi_tune,
            levels=levels_tune,
        )

        return float(mean_absolute_error(cv["y"], cv[pred_col]))

    return objective_fn

In [ ]:
if spec.name == "RIDGE_EXOG_ONLY":
    objective = make_objective_ridge_sklearn_cv(
        ts_train,
        cv=5,
    )
else:
    objective = make_objective_nixtla_cv(
        ts_train,
        build_mlf_fn=lambda p, _bf=spec.build_from_params: _bf(p),
        run_cv_block_fn=run_cross_validation_block,
        pred_col=spec.pred_col,
        h=h,
        step_size=int(spec.tune_step_size),
        tune_cv_windows=int(spec.tune_cv_windows),
        pi_tune=spec.pi_tune,
        levels_tune=spec.levels_tune,
    )

## PI conformal

In [95]:
pi = PredictionIntervals(
    h=h,
    n_windows=PI_WINDOWS,
    method="conformal_distribution",
)

# Backtesting

In [96]:
def run_cross_validation_block(
    *,
    ts,
    mlf,
    h,
    step_size,
    partitions,
    pi,
    levels,
):
    return mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
        static_features=[],
    )

# Run

In [97]:
# ============================================================
# RUN (agnostic) — ne connaît aucun modèle
# ============================================================

from dataclasses import dataclass
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta


@dataclass
class ModelSpec:
    name: str
    # Build model for production backtest
    build_fixed: Optional[Callable[[], Any]] = None
    build_from_params: Optional[Callable[[Dict[str, Any]], Any]] = None

    # Tuning (optional)
    tunable: bool = False
    param_space: Optional[Dict[str, Iterable[Any]]] = None
    search: str = "random"          # "random" | "grid"
    n_iter: int = 50
    seed: int = 0

    # Objective tuning via Nixtla mini-CV
    pred_col: Optional[str] = None
    tune_cv_windows: int = 6
    tune_step_size: int = 1
    pi_tune: Any = None
    levels_tune: Any = None


def run_models_backtest_with_block_retuning(
    ts: pd.DataFrame,
    *,
    freq: str,
    h: int,
    exp_start,
    exp_end,
    step_size: int,
    levels,
    pi_windows: int,
    tune_every_months: int,
    min_train_n: Optional[int],
    model_specs: Sequence[ModelSpec],
):
    """
    RUN agnostic:
      - crée des blocs (tune_every_months)
      - pour chaque bloc: construit ts_train (anti-fuite), tune chaque modèle tunable via tune_model(),
        puis exécute run_cross_validation_block() sur ts_blk
      - concatène tout dans bkt_all + meta
    """

    # 1) Préparation dates + anti-fuite
    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)
    ts = _prep_monthly_ts(ts, exp_end=exp_end)

    # 2) partitions globales
    cutoff_start_all = exp_start - relativedelta(months=h)
    cutoff_end_all   = exp_end   - relativedelta(months=h)
    partitions_all   = _n_windows_monthly(cutoff_start_all, cutoff_end_all)

    # 3) Prediction Intervals (prod CV)
    pi = PredictionIntervals(h=h, n_windows=pi_windows, method="conformal_distribution")

    # 4) construire blocs
    blocks: List[Tuple[Any, int]] = []
    remaining = int(partitions_all)
    cur = cutoff_start_all
    while remaining > 0:
        n = min(int(tune_every_months), int(remaining))
        blocks.append((cur, n))
        cur = cur + relativedelta(months=n)
        remaining -= n

    all_bkts: List[pd.DataFrame] = []
    tuning_history: List[Dict[str, Any]] = []

    # 5) boucle blocs
    for block_idx, (cutoff_start_blk, partitions_blk) in enumerate(blocks, start=1):

        # train pour tuning (anti-fuite)
        ts_train = ts[ts["ds"] <= cutoff_start_blk].copy()
        if min_train_n is not None and len(ts_train) < int(min_train_n):
            continue

        # slice bloc pour backtest "prod"
        ts_blk, cutoff_end_blk, ds_end_blk = _slice_cv_block(ts, cutoff_start_blk, partitions_blk, h)

        # boucle modèles
        for spec in model_specs:

            used_params = None
            tune_score = None

            if spec.tunable:
                if spec.build_from_params is None:
                    raise ValueError(f"{spec.name}: build_from_params requis quand tunable=True")
                if spec.param_space is None:
                    raise ValueError(f"{spec.name}: param_space requis quand tunable=True")
                if spec.pred_col is None:
                    raise ValueError(f"{spec.name}: pred_col requis quand tunable=True")

                # objective via TON runner
                objective = make_objective_nixtla_cv(
                    ts_train,
                    build_mlf_fn=lambda p, _bf=spec.build_from_params: _bf(p),
                    run_cv_block_fn=run_cross_validation_block,
                    pred_col=spec.pred_col,
                    h=h,
                    step_size=int(spec.tune_step_size),
                    tune_cv_windows=int(spec.tune_cv_windows),
                    pi_tune=spec.pi_tune,
                    levels_tune=spec.levels_tune,
                )

                res = tune_model(
                    ts_train,
                    objective_fn=objective,
                    param_space=spec.param_space,
                    search=spec.search,
                    n_iter=int(spec.n_iter),
                    seed=int(spec.seed),
                    min_train_n=min_train_n,
                    return_trials=False,
                )

                used_params = res.best_params
                tune_score = res.best_score

                tuning_history.append({
                    "block": block_idx,
                    "cutoff_start": cutoff_start_blk,
                    "partitions": int(partitions_blk),
                    "model_name": spec.name,
                    "best_params": used_params,
                    "tune_score": tune_score,
                })

                if used_params is None or not np.isfinite(tune_score):
                    continue

                mlf = spec.build_from_params(used_params)

            else:
                if spec.build_fixed is None:
                    raise ValueError(f"{spec.name}: build_fixed requis quand tunable=False")
                mlf = spec.build_fixed()

            # backtest prod sur bloc
            bkt_blk = run_cross_validation_block(
                ts=ts_blk,
                mlf=mlf,
                h=h,
                step_size=step_size,
                partitions=int(partitions_blk),
                pi=pi,
                levels=list(levels),
            ).copy()

            bkt_blk["model_name"] = spec.name
            bkt_blk["tune_block"] = block_idx
            bkt_blk["params_used"] = None if used_params is None else str(used_params)
            bkt_blk["tune_score"] = tune_score
            all_bkts.append(bkt_blk)

    if not all_bkts:
        return pd.DataFrame(), {"error": "Aucun bloc produit"}

    bkt_all = (
        pd.concat(all_bkts, ignore_index=True)
          .sort_values(["model_name", "unique_id", "ds", "cutoff"])
          .reset_index(drop=True)
    )

    meta = {
        "h": h,
        "freq": freq,
        "step_size": step_size,
        "exp_start": exp_start,
        "exp_end": exp_end,
        "partitions": int(partitions_all),
        "pi_windows": int(pi_windows),
        "levels": list(levels),
        "tune_every_months": int(tune_every_months),
        "tuning_history": tuning_history,
    }
    return bkt_all, meta

In [98]:
# -----------------------------
# Paramètres globaux
# -----------------------------
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
FREQ = "MS"

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")

TUNE_EVERY_MONTHS = 36
MIN_TRAIN_N = 36
TUNE_CV = 6
SEED = 0

# -----------------------------
# Grilles (exemples)
# -----------------------------
ALPHA_GRID = np.logspace(-3, 3, 25)

PARAM_GRID_LGBM = {
    "n_estimators": [200, 500, 1000],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [15, 31, 63],
    "max_depth": [-1, 4, 6, 8],
    "min_child_samples": [10, 20, 50],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "reg_lambda": [0.0, 0.1, 1.0],
}

In [102]:
# -----------------------------
# Specs modèles (ICI tu "mets en place les modèles")
# -----------------------------
model_specs = [
    # Ridge retuné
    ModelSpec(
        name="RIDGE_EXOG_ONLY",
        tunable=True,
        build_from_params=lambda p: MLF_MODELS["RIDGE_EXOG_ONLY"](FREQ, **p),
        param_space={"alpha": list(ALPHA_GRID)},
        search="grid",
        pred_col="Ridge",              # ⚠️ adapte au nom de colonne CV
        tune_cv_windows=TUNE_CV,
        seed=SEED,
    ),

    # LGBM retuné
    ModelSpec(
        name="LGBM_EXOG_ONLY",
        tunable=True,
        build_from_params=lambda p: MLF_MODELS["LGBM_EXOG_ONLY"](FREQ, **p),
        param_space=PARAM_GRID_LGBM,
        search="random",
        n_iter=100,
        pred_col="LGBM",
        tune_cv_windows=6,
        seed=SEED,
    ),

    # Modèles fixes
    ModelSpec(
        name="LR_EXOG_ONLY",
        build_fixed=lambda: MLF_MODELS["LR_EXOG_ONLY"](FREQ),
        tunable=False,
    )
]

In [ ]:
# -----------------------------
# Lancement
# -----------------------------
bkt_all, meta = run_models_backtest_with_block_retuning(
    ts_lr,
    freq=FREQ,
    h=H,
    exp_start=EXP_START,
    exp_end=EXP_END,
    step_size=STEP_SIZE,
    levels=LEVELS,
    pi_windows=PI_WINDOWS,
    tune_every_months=TUNE_EVERY_MONTHS,
    min_train_n=MIN_TRAIN_N,
    model_specs=model_specs,
)

In [ ]:
list(MLF_MODELS.keys())

['LR_EXOG_ONLY', 'LGBM_EXOG_ONLY', 'RIDGE_EXOG_ONLY']

# Transformer le Backtesting

In [ ]:
bkt_all_final = (
    bkt_all
    .groupby("model_name", group_keys=False)
    .apply(lambda df: finalize_backtest_run(df, exp_start=EXP_START, exp_end=EXP_END, verbose=False))
    .reset_index(drop=True)
)

bkt_all_final[["model_name","unique_id","ds","cutoff"]].head()

C:\Users\Mita\AppData\Local\Temp\ipykernel_10372\983196062.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: finalize_backtest_run(df, exp_start=EXP_START, exp_end=EXP_END, verbose=False))


,model_name,unique_id,ds,cutoff
0,LR_EXOG_ONLY,UNRATE,1990-01-01,1989-12-01
1,LR_EXOG_ONLY,UNRATE,1990-02-01,1990-01-01
2,LR_EXOG_ONLY,UNRATE,1990-03-01,1990-02-01
3,LR_EXOG_ONLY,UNRATE,1990-04-01,1990-03-01
4,LR_EXOG_ONLY,UNRATE,1990-05-01,1990-04-01


In [ ]:
bkt_score = bkt_all_final.copy()
bkt_score["ds"] = pd.to_datetime(bkt_score["ds"], errors="coerce")

# enlever timezone si jamais
if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):
    bkt_score["ds"] = bkt_score["ds"].dt.tz_convert(None)

bkt_score = bkt_score.dropna(subset=["ds"])
bkt_score = bkt_score[(bkt_score["ds"] >= EXP_START) & (bkt_score["ds"] <= EXP_END)].reset_index(drop=True)

bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-end"]

bkt_score["partition"] = pd.cut(bkt_score["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
bkt_score = bkt_score.dropna(subset=["partition"]).reset_index(drop=True)

print(bkt_score[["ds","cutoff","partition"]].head(10))
print(bkt_score["partition"].value_counts().sort_index())

          ds     cutoff  partition
0 1990-01-01 1989-12-01  1990-1999
1 1990-02-01 1990-01-01  1990-1999
2 1990-03-01 1990-02-01  1990-1999
3 1990-04-01 1990-03-01  1990-1999
4 1990-05-01 1990-04-01  1990-1999
5 1990-06-01 1990-05-01  1990-1999
6 1990-07-01 1990-06-01  1990-1999
7 1990-08-01 1990-07-01  1990-1999
8 1990-09-01 1990-08-01  1990-1999
9 1990-10-01 1990-09-01  1990-1999
partition
1990-1999    240
2000-2008    216
2009-2019    264
2020-end     136
Name: count, dtype: int64


C:\Users\Mita\AppData\Local\Temp\ipykernel_10372\3658633496.py:5: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(bkt_score["ds"]):


# Building Leaderbord

In [ ]:
tmp = bkt_score.copy()

models = ["LR", "RIDGE", "LGBM"]

# 1) S'assurer que lower <= upper (sécurité)
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"
    tmp[[lo, hi]] = np.sort(tmp[[lo, hi]].to_numpy(), axis=1)

# 2) Wide -> Long + features de scoring
rows = []
for m in models:
    lo = f"{m}-lo-95"
    hi = f"{m}-hi-95"

    s = tmp[["unique_id", "ds", "cutoff", "y", "partition"]].copy()
    s["model_label"] = m
    s["model_name"]  = m

    s["forecast"] = tmp[m]
    s["lower"]    = tmp[lo]
    s["upper"]    = tmp[hi]

    s["abs_err"]   = (s["y"] - s["forecast"]).abs()
    s["covered"]   = ((s["y"] >= s["lower"]) & (s["y"] <= s["upper"])).astype(int)
    s["int_width"] = (s["upper"] - s["lower"]).abs()

    rows.append(s)

long_sc = pd.concat(rows, ignore_index=True)

# 3) FIX pandas: partition en string + groupby reset_index
long_sc["partition"] = long_sc["partition"].astype(str)
long_sc = long_sc.loc[:, ~long_sc.columns.duplicated()]

score_df = (
    long_sc
    .groupby(["unique_id", "model_label", "model_name", "partition"], observed=True)
    .agg(
        mae=("abs_err", "mean"),
        coverage=("covered", "mean"),
        width=("int_width", "mean"),
        n=("y", "size"),
    )
    .reset_index()
)

# 4) Top 3 par partition
leaderboard = (
    score_df.sort_values(
        by=["partition", "mae", "coverage", "width"],
        ascending=[True, True, False, True],
    )
    .groupby("partition", as_index=False)
    .head(3)
)

print(score_df.sort_values(["partition", "mae"]).head(20))
print("\nTop 3 par partition:")
print(leaderboard)

KeyError: "None of [Index(['LGBM-lo-95', 'LGBM-hi-95'], dtype='object')] are in the [columns]"

# MLFLOW

In [ ]:
import mlflow

# =====================================================
# 0) (Optionnel mais recommandé) Tracking URI
# =====================================================
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# =====================================================
# 1) Set experiment
# =====================================================
experiment_name = "Multivariate_experiment_tuning"
mlflow.set_experiment(experiment_name)

# =====================================================
# 2) DataFrame à logger
# =====================================================
df_log = score_df.copy()  # ou leaderboard_global

# =====================================================
# 3) Logging loop
# =====================================================
for idx, row in df_log.iterrows():
    
    partition = row["partition"] if "partition" in df_log.columns else "global"
    run_name = f"{row['model_label']}_partition_{partition}"
    
    with mlflow.start_run(run_name=run_name):
        
        # ======================
        # PARAMETERS
        # ======================
        mlflow.log_param("model_label", row["model_label"])
        mlflow.log_param("model_name", row["model_name"])
        mlflow.log_param("partition", partition)
        mlflow.log_param("horizon", 12)
        mlflow.log_param("lags", list(range(1, 25)))
        mlflow.log_param("date_features", ["year", "month", "quarter"])
        mlflow.log_param("conformal_method", "conformal_distribution")
        
        # ======================
        # METRICS
        # ======================
        mlflow.log_metric("mae", float(row["mae"]))
        mlflow.log_metric("coverage", float(row["coverage"]))
        mlflow.log_metric("width", float(row["width"]))
        
        # ======================
        # TAGS (optionnel mais pro)
        # ======================
        mlflow.set_tag("project", "UNRATE_forecasting")
        mlflow.set_tag("framework", "MLForecast")
        mlflow.set_tag("model_family", row["model_name"])

print("Logging terminé.")

🏃 View run LGBM_partition_1990-1999 at: http://127.0.0.1:5000/#/experiments/7/runs/c56768866d6f484eb1a650aa1f707bd9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2000-2008 at: http://127.0.0.1:5000/#/experiments/7/runs/00b20709239b49fbb5d9c5830852b326
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2009-2019 at: http://127.0.0.1:5000/#/experiments/7/runs/5f739ef04ff74e669be9da5471877df5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LGBM_partition_2020-end at: http://127.0.0.1:5000/#/experiments/7/runs/b6e34595f5c248cea352f3559b68a550
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LR_partition_1990-1999 at: http://127.0.0.1:5000/#/experiments/7/runs/3cc4a6ae38ca41af8dd18e8669032ea4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7
🏃 View run LR_partition_2000-2008 at: http://127.0.0.1:5000/#/experiments/7/runs/3bfeed74e1a14cb0bf859c2b45eb5f6a
🧪 View ex

# Vérif

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# =========================================================
# MAE pour LR (global + par partition) à partir de bkt_lr_final
# Colonnes attendues: unique_id, ds, y, LR
# =========================================================

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")

df = bkt_lr_final.copy()

# ds propre
df["ds"] = pd.to_datetime(df["ds"], errors="coerce")
if pd.api.types.is_datetime64tz_dtype(df["ds"]):
    df["ds"] = df["ds"].dt.tz_convert(None)

# garder lignes valides
df = df.dropna(subset=["ds", "y", "LR"]).copy()
df = df[(df["ds"] >= EXP_START) & (df["ds"] <= EXP_END)].reset_index(drop=True)

# -----------------------------
# MAE global
# -----------------------------
mae_all = mean_absolute_error(df["y"], df["LR"])
print("✅ MAE moyen (LR) — GLOBAL :", float(mae_all))

# -----------------------------
# MAE par partition macro (based on ds)
# -----------------------------
bins = pd.to_datetime(["1990-01-01","2000-01-01","2009-01-01","2020-01-01","2025-09-01"])
labels = ["1990-1999", "2000-2008", "2009-2019", "2020-fin"]

df["partition"] = pd.cut(df["ds"], bins=bins, labels=labels, right=False, include_lowest=True)
df = df.dropna(subset=["partition"]).copy()

mae_by_partition = (
    df.assign(abs_err=np.abs(df["y"] - df["LR"]))
      .groupby("partition")["abs_err"]
      .mean()
      .sort_index()
)

n_by_partition = df.groupby("partition").size().sort_index()

print("\n✅ MAE moyen (LR) — PAR PARTITION :")
print(mae_by_partition)

print("\n✅ N obs — PAR PARTITION :")
print(n_by_partition)

# (optionnel) MAE pondéré (≈ MAE global)
mae_weighted = float((mae_by_partition * n_by_partition).sum() / n_by_partition.sum())
print("\n✅ MAE moyen (LR) — pondéré partitions :", mae_weighted)

✅ MAE moyen (LR) — GLOBAL : 0.6799697179008874

✅ MAE moyen (LR) — PAR PARTITION :
partition
1990-1999    0.421220
2000-2008    0.377789
2009-2019    0.597483
2020-fin     1.776643
Name: abs_err, dtype: float64

✅ N obs — PAR PARTITION :
partition
1990-1999    120
2000-2008    108
2009-2019    132
2020-fin      68
dtype: int64

✅ MAE moyen (LR) — pondéré partitions : 0.6799697179008874


C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:17: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df["ds"]):
C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("partition")["abs_err"]
C:\Users\Mita\AppData\Local\Temp\ipykernel_16388\3500170289.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  n_by_partition = df.groupby("partition").size().sort_index()
